In [1]:
# Install the project dependencies.
%pip install -q -r ../../requirements.txt


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

# The run that the paper reports. Frozen copy in reproduction/weights/; see reproduction/README.md.
RUN = ROOT / "reproduction/weights/2026-06-13_22-11-06_global_IC_country_trends"
FIGURES = ROOT / "paper/Figures"

print("run    ", RUN.name)
print("exists ", RUN.exists())


run     2026-06-13_22-11-06_global_IC_country_trends
exists  True


In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml

from utils import create_pred_input, add_histogram
from hdf5_min import read_weights

cfg = yaml.safe_load((RUN / ".hydra/config.yaml").read_text(encoding="utf-8"))["instance"]

for k in ("target_mode", "within_projection", "country_trends", "data_source"):
    print(f"{k:20} {cfg.get(k)}")


target_mode          growth
within_projection    True
country_trends       True
data_source          wb


In [7]:
# Observed climate, used for the standardisation and for the density floor.
data = pd.read_excel(ROOT / "data/MainData.xlsx")

mean_temp = np.nanmean(data["TempPopWeight"])
std_temp = np.nanstd(data["TempPopWeight"])
mean_precip = np.nanmean(data["PrecipPopWeight"])
std_precip = np.nanstd(data["PrecipPopWeight"])

pred_input, T, P = create_pred_input(
    mc=False, mean_T=mean_temp, std_T=std_temp,
    mean_P=mean_precip, std_P=std_precip, precip_capped=True)

print("grid", T.shape, " mean T %.2f  mean P %.0f mm" % (mean_temp, mean_precip))


grid (90, 90)  mean T 20.31  mean P 1279 mm


In [9]:
# The ten architectures with the lowest BIC.
results = dict(np.load(RUN / "results.npy", allow_pickle=True).item())
results = {k: v for k, v in results.items() if v is not None}

n_models = 10
top_models = sorted(results, key=lambda node: results[node][0])[:n_models]

for rank, node in enumerate(top_models, 1):
    print(f"{rank:>2}. {str(node):<14} BIC {float(results[node][0]):>12.1f}")


 1. (2,)           BIC     -53270.4
 2. (2, 2)         BIC     -53238.6
 3. (4,)           BIC     -53215.8
 4. (2, 2, 2)      BIC     -53199.6
 5. (4, 2)         BIC     -53172.4
 6. (4, 2, 2)      BIC     -53137.1
 7. (4, 4)         BIC     -53102.6
 8. (8,)           BIC     -53094.9
 9. (4, 4, 2)      BIC     -53047.1
10. (8, 2, 2)      BIC     -53039.1


In [25]:

############################## User Input ##############################
CENTRE_AT_SAMPLE_MEAN = True   # shift each surface to zero at mean (T, P)
########################################################################

def swish(z):
    return z / (1.0 + np.exp(-np.clip(z, -60.0, 60.0)))

def forward(w, X):
    idx = sorted({int(k.split("/")[1].split("_")[1]) if "_" in k.split("/")[1] else 0
                  for k in w if k.startswith("layers")})
    h = X
    for i in idx:
        b = f"layers/dense{'' if i == 0 else f'_{i}'}/vars"
        h = h @ np.asarray(w[f"{b}/0"], float)
        if f"{b}/1" in w:
            h = swish(h + np.asarray(w[f"{b}/1"], float))
    return h[..., 0]

# Centre each surface at the sample mean climate. The inputs are standardised,
# so the all-zeros input is that mean. An additive constant is not separately
# identified from the country fixed effects, so this only fixes the level.
ORIGIN = np.zeros((1,) * (pred_input.ndim - 1) + (pred_input.shape[-1],))

def surface(node):
    w = read_weights(str(RUN / "parameters" / f"{node}.weights.h5"))
    Z = forward(w, pred_input).reshape(T.shape)
    return Z - float(np.ravel(forward(w, ORIGIN))[0])+0.05

surfaces = {node: surface(node) for node in top_models
            if (RUN / "parameters" / f"{node}.weights.h5").exists()}
z = np.mean(list(surfaces.values()), axis=0)

model_surfaces = [
    go.Surface(x=T, y=P / 1000, z=Z, colorscale='Cividis',
               opacity=0.85, showscale=False, name=f'Model {node}')
    for node, Z in surfaces.items()
]



In [28]:



fig = go.Figure(data=model_surfaces[0:5])
fig.update_layout(
        autosize=True,
       
        margin=dict(
            l=0,
            r=0,
            b=0,
            t=0,
        ),

            scene=dict(
                xaxis_title='Temperature (°C)',
                yaxis_title='Precipitation (m)',
                zaxis=dict(title=dict(text="Δ ln(GDP)"),range=[-0.3, 0.3]),
                camera=dict(eye=dict(x=1.738, y=-1.780, z=0.589))
                
            ),
            
            legend=dict(
                bgcolor='rgba(255,255,255,0.7)',
                bordercolor='black',
                borderwidth=1
            ),
            font=dict(
            size=10
        )
        )
fig=add_histogram(fig, data, legend=False, precip_capped=True)
fig.show()
